# Rain in Australia: Predicting `RainTomorrow`

**Goal:** predict whether it will rain tomorrow at an Australian weather station, using today's weather observations.

This notebook covers:

1. Loading and getting to know the dataset
2. Data cleaning (target, columns with many missing values, imputation)
3. Train / validation / test split
4. Exploratory data analysis (EDA) and feature importance
5. Training and comparing several models: Logistic Regression, Decision Tree, Random Forest, XGBoost
6. Choosing the best model, retraining it on train + validation and evaluating it on the test set
7. Saving the model and checking the saved file (section 8)
8. Serving the model as a web service with FastAPI: `predict.py`, `test.py` (section 9)
9. Packaging the service with Docker (section 10)
10. Deploying it to AWS Elastic Beanstalk, testing it, and deleting the environment (section 11)
11. Other ways to deploy a model on AWS (section 12)

**Dataset:** [Rain in Australia (Kaggle)](https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package), file `weatherAUS.csv`, about 10 years of daily observations from 49 locations.

## 0. Environment setup

The project uses [`uv`](https://docs.astral.sh/uv/) to manage the Python environment and dependencies.

```bash
# 1. Install uv (see https://docs.astral.sh/uv/getting-started/installation/)
# 2. Check the installation
uv --version
# 3. Create the environment and install the dependencies from pyproject.toml / uv.lock
uv sync
# (dependencies were added with e.g. `uv add pandas scikit-learn xgboost`,
#  and for the web service: `uv add fastapi uvicorn requests`)
```

Then select the `.venv` kernel for this notebook.

In [1]:
import pickle

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import accuracy_score, roc_auc_score, mutual_info_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import xgboost as xgb

## 1. Loading the data and first look

In [2]:
df = pd.read_csv('weatherAUS.csv')
print(df.shape)
df.head()

(275410, 24)


,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RISK_MM,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,0.0,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,0.0,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,0.0,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,1.0,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,0.2,No


In [3]:
df.dtypes

Date                 str
Location             str
MinTemp          float64
MaxTemp          float64
Rainfall         float64
Evaporation      float64
Sunshine         float64
WindGustDir          str
WindGustSpeed    float64
WindDir9am           str
WindDir3pm           str
WindSpeed9am     float64
WindSpeed3pm     float64
Humidity9am      float64
Humidity3pm      float64
Pressure9am      float64
Pressure3pm      float64
Cloud9am         float64
Cloud3pm         float64
Temp9am          float64
Temp3pm          float64
RainToday            str
RISK_MM          float64
RainTomorrow         str
dtype: object

Short description of the columns:

| Column | Meaning |
|---|---|
| `Date`, `Location` | Observation date and weather station |
| `MinTemp`, `MaxTemp`, `Temp9am`, `Temp3pm` | Temperatures (°C) |
| `Rainfall` | Rainfall during the day (mm) |
| `Evaporation`, `Sunshine` | Evaporation (mm) and hours of bright sunshine |
| `WindGustDir`, `WindGustSpeed` | Direction and speed (km/h) of the strongest wind gust |
| `WindDir9am/3pm`, `WindSpeed9am/3pm` | Wind direction and speed at 9am / 3pm |
| `Humidity9am/3pm`, `Pressure9am/3pm`, `Cloud9am/3pm` | Humidity (%), pressure (hPa), cloud cover (oktas) |
| `RainToday` | `Yes` if rainfall today > 1 mm |
| `RISK_MM` | Amount of rain **tomorrow** (mm) |
| `RainTomorrow` | **Target:** `Yes` if it rains tomorrow |

In [4]:
df.describe().round(3)

,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RISK_MM
count,270732.000,270779.000,267316.000,115284.000,104284.000,255145.000,269462.000,264722.000,269181.000,263949.000,245025.000,245048.000,148891.000,142702.000,270776.000,265428.000,267317.000
mean,12.096,23.264,2.339,5.494,7.629,39.993,14.052,18.534,69.078,51.464,1017.662,1015.230,4.779,4.735,16.939,21.752,2.339
std,6.432,7.135,8.608,4.138,3.776,13.501,8.823,8.677,19.386,21.027,7.099,7.055,2.896,2.771,6.535,7.001,8.608
min,-8.700,-4.100,0.000,0.000,0.000,2.000,0.000,0.000,0.000,0.000,978.600,972.000,0.000,0.000,-6.200,-5.100,0.000
25%,7.500,18.000,0.000,2.800,4.900,31.000,7.000,13.000,57.000,36.000,1012.900,1010.400,2.000,2.000,12.200,16.700,0.000
50%,11.900,22.700,0.000,4.800,8.500,39.000,13.000,17.000,70.000,52.000,1017.600,1015.200,6.000,5.000,16.700,21.200,0.000
75%,16.800,28.200,0.600,7.400,10.600,48.000,19.000,24.000,84.000,66.000,1022.500,1020.100,7.000,7.000,21.500,26.500,0.600
max,33.900,49.600,474.000,138.700,14.500,154.000,87.000,87.000,100.000,100.000,1042.500,1040.300,9.000,9.000,41.000,48.200,474.000


### Missing values

There are many missing values. Let's look at the percentage of missing values per column.

In [5]:
missing_pct = df.isna().mean().mul(100).sort_values(ascending=False)
missing_pct.round(1)

Sunshine         62.1
Evaporation      58.1
Cloud3pm         48.2
Cloud9am         45.9
Pressure9am      11.0
Pressure3pm      11.0
WindDir9am        7.8
WindGustDir       7.4
WindGustSpeed     7.4
WindDir3pm        4.5
Humidity3pm       4.2
WindSpeed3pm      3.9
Temp3pm           3.6
Rainfall          2.9
RainToday         2.9
RISK_MM           2.9
RainTomorrow      2.9
Humidity9am       2.3
WindSpeed9am      2.2
MinTemp           1.7
Temp9am           1.7
MaxTemp           1.7
Location          0.0
Date              0.0
dtype: float64

`Sunshine`, `Evaporation`, `Cloud3pm` and `Cloud9am` are missing in roughly 45–60% of the rows, which is too many to impute reliably.

The target `RainTomorrow` is also missing in about 3% of the rows.

## 2. Data cleaning

### 2.1 Target variable

We can't train on rows with an unknown label, so we drop them and encode the target as `1` (Yes) / `0` (No).

In [6]:
print(df.RainTomorrow.unique())

df_clean = df.dropna(subset=['RainTomorrow']).copy()
df_clean['RainTomorrow'] = df_clean['RainTomorrow'].map({'Yes': 1, 'No': 0})

print(df_clean.RainTomorrow.unique())

<StringArray>
['No', 'Yes', nan]
Length: 3, dtype: str
[0 1]


### 2.2 Dropping columns

- `Sunshine`, `Evaporation`, `Cloud3pm`, `Cloud9am`: more than 45% missing values.
- `Date`: we drop it to keep things simple. (A possible improvement is to extract the month as a feature to capture seasonality.)

In [7]:
df_clean = df_clean.drop(columns=['Sunshine', 'Evaporation', 'Cloud3pm', 'Cloud9am', 'Date'])
df_clean.head()

,Location,MinTemp,MaxTemp,Rainfall,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Temp9am,Temp3pm,RainToday,RISK_MM,RainTomorrow
0,Albury,13.4,22.9,0.6,W,44.0,W,WNW,20.0,24.0,71.0,22.0,1007.7,1007.1,16.9,21.8,No,0.0,0
1,Albury,7.4,25.1,0.0,WNW,44.0,NNW,WSW,4.0,22.0,44.0,25.0,1010.6,1007.8,17.2,24.3,No,0.0,0
2,Albury,12.9,25.7,0.0,WSW,46.0,W,WSW,19.0,26.0,38.0,30.0,1007.6,1008.7,21.0,23.2,No,0.0,0
3,Albury,9.2,28.0,0.0,NE,24.0,SE,E,11.0,9.0,45.0,16.0,1017.6,1012.8,18.1,26.5,No,1.0,0
4,Albury,17.5,32.3,1.0,W,41.0,ENE,NW,7.0,20.0,82.0,33.0,1010.8,1006.0,17.8,29.7,No,0.2,0


In [8]:
missing_pct = df_clean.isna().mean().mul(100).sort_values(ascending=False)
missing_pct.round(1)

Pressure9am      10.8
Pressure3pm      10.8
WindDir9am        7.6
WindGustDir       6.8
WindGustSpeed     6.7
WindDir3pm        4.3
WindSpeed3pm      3.7
Humidity3pm       3.0
Temp3pm           2.5
WindSpeed9am      1.9
Humidity9am       1.1
RainToday         1.0
Rainfall          1.0
MinTemp           0.5
Temp9am           0.5
MaxTemp           0.4
Location          0.0
RISK_MM           0.0
RainTomorrow      0.0
dtype: float64

### 2.3 Categorical and numerical columns

We keep the target (`RainTomorrow`) out of the feature lists.

In [9]:
target = 'RainTomorrow'

categorical = list(df_clean.select_dtypes(exclude='number').columns)
numerical = [c for c in df_clean.select_dtypes(include='number').columns if c != target]

print('categorical:', categorical)
print('numerical:  ', numerical)

categorical: ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday']
numerical:   ['MinTemp', 'MaxTemp', 'Rainfall', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Temp9am', 'Temp3pm', 'RISK_MM']


## 3. Train / validation / test split

We split the data 60% / 20% / 20%:

- `df_train`: training the models
- `df_valid`: comparing models and tuning parameters
- `df_test`: final evaluation of the chosen model
- `df_full_train` = `df_train` + `df_valid`: training the final model

In [10]:
df_full_train, df_test = train_test_split(df_clean, test_size=0.2, random_state=1)
df_train, df_valid = train_test_split(df_full_train, test_size=0.25, random_state=1)

len(df_full_train), len(df_train), len(df_valid), len(df_test)

(213853, 160389, 53464, 53464)

### 3.1 Filling missing values

- **Numerical columns:** filled with the median.
- **Categorical columns:** filled with a new category `'Missing'`.

The medians are computed **on the training data only** and then applied to every split. This way no information from the validation/test data leaks into the training process.

In [11]:
train_medians = df_train[numerical].median()
print(train_medians)


def fill_missing(df):
    df = df.copy()
    df[numerical] = df[numerical].fillna(train_medians)
    df[categorical] = df[categorical].fillna('Missing')
    return df


df_full_train = fill_missing(df_full_train)
df_train = fill_missing(df_train)
df_valid = fill_missing(df_valid)
df_test = fill_missing(df_test)

# number of remaining missing values in every split (should all be 0)
pd.Series({
    'full_train': df_full_train.isna().sum().sum(),
    'train': df_train.isna().sum().sum(),
    'valid': df_valid.isna().sum().sum(),
    'test': df_test.isna().sum().sum(),
})

MinTemp            11.9
MaxTemp            22.7
Rainfall            0.0
WindGustSpeed      39.0
WindSpeed9am       13.0
WindSpeed3pm       17.0
Humidity9am        70.0
Humidity3pm        52.0
Pressure9am      1017.7
Pressure3pm      1015.2
Temp9am            16.7
Temp3pm            21.2
RISK_MM             0.0
dtype: float64


full_train    0
train         0
valid         0
test          0
dtype: int64

In [12]:
df_train.head(10)

,Location,MinTemp,MaxTemp,Rainfall,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Temp9am,Temp3pm,RainToday,RISK_MM,RainTomorrow
244003,Hobart,13.2,19.7,0.8,ESE,26.0,SSW,SE,9.0,13.0,84.0,61.0,1023.8,1022.2,14.1,18.6,No,0.0,0
104051,Ballarat,2.9,9.8,4.6,SSE,57.0,SSW,SE,19.0,35.0,98.0,98.0,1009.6,1011.8,4.7,7.3,Yes,3.6,1
203164,Woomera,16.7,31.3,0.0,SE,57.0,SSE,SSE,20.0,35.0,58.0,26.0,1013.0,1011.4,21.1,30.8,No,0.0,0
115382,Sale,14.1,27.7,0.0,SW,35.0,S,ESE,6.0,11.0,72.0,67.0,1011.7,1012.0,21.2,23.9,No,0.0,0
15882,Cobar,13.4,25.2,0.0,NE,33.0,ENE,ENE,17.0,17.0,63.0,41.0,1028.7,1024.0,18.4,23.7,No,0.0,0
225156,PerthAirport,6.8,25.5,0.0,SW,35.0,ENE,WSW,7.0,19.0,52.0,35.0,1023.0,1019.3,18.8,25.0,No,0.0,0
220196,PearceRAAF,16.3,27.2,0.0,SSW,56.0,S,SW,39.0,33.0,43.0,38.0,1012.4,1011.1,22.2,24.2,No,0.0,0
200529,Woomera,18.5,29.8,0.0,SSE,35.0,SSE,NW,24.0,17.0,46.0,27.0,1013.6,1013.5,22.5,29.2,No,2.2,1
45431,Penrith,15.8,30.2,0.0,SW,41.0,Missing,W,0.0,6.0,85.0,67.0,1017.7,1015.2,20.8,25.6,No,10.6,1
216640,PearceRAAF,21.8,34.6,0.0,SW,39.0,SSW,SW,17.0,26.0,51.0,40.0,1010.5,1007.8,27.9,32.2,No,0.0,0


In [13]:
y_full_train = df_full_train[target].values
y_train = df_train[target].values
y_valid = df_valid[target].values
y_test = df_test[target].values

## 4. Exploratory data analysis

All the analysis below is done on `df_full_train` so that the test set stays unseen.

### 4.1 Target distribution

In [14]:
df_full_train[target].value_counts(normalize=True)

RainTomorrow
0    0.779961
1    0.220039
Name: proportion, dtype: float64

The classes are imbalanced: it rains on about 22% of the days. A model that always predicts "No rain" would already get about 78% accuracy, so we will mainly use **ROC AUC** to compare models.

### 4.2 Categorical features

In [15]:
df_full_train[categorical].nunique()

Location       49
WindGustDir    17
WindDir9am     17
WindDir3pm     17
RainToday       3
dtype: int64

In [16]:
df_full_train.WindDir9am.unique()

<StringArray>
[    'ESE', 'Missing',       'W',      'SE',     'NNE',      'NW',     'WSW',
     'ENE',     'NNW',       'E',     'SSE',      'NE',     'SSW',       'N',
     'WNW',      'SW',       'S']
Length: 17, dtype: str

### 4.3 Rain rate per category

For every category we compute:

- `mean`: rain rate in this group
- `diff = mean - global_rain`: positive means rain is more likely than average
- `risk = mean / global_rain`: > 1 means rain is more likely than average

In [17]:
global_rain = df_full_train[target].mean()
print(f'Global rain rate: {global_rain:.3f}')

for c in categorical:
    print(c)
    df_group = df_full_train.groupby(c)[target].agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_rain
    df_group['risk'] = df_group['mean'] / global_rain
    display(df_group)
    print()

Global rain rate: 0.220
Location


,mean,count,diff,risk
Location,,,,
Adelaide,0.215553,3099,-0.004486,0.979615
Albany,0.301101,4540,0.081062,1.368400
Albury,0.198611,4607,-0.021428,0.902616
AliceSprings,0.068439,4588,-0.151600,0.311033
BadgerysCreek,0.197553,4414,-0.022486,0.897810
Ballarat,0.251311,4576,0.031272,1.142121
Bendigo,0.176964,4532,-0.043075,0.804238
Brisbane,0.224534,4663,0.004495,1.020426
Cairns,0.312570,4495,0.092531,1.420519



WindGustDir


,mean,count,diff,risk
WindGustDir,,,,
E,0.144723,13474,-0.075316,0.657716
ENE,0.161266,12160,-0.058773,0.732899
ESE,0.165107,11223,-0.054932,0.750355
Missing,0.255615,14514,0.035576,1.161682
N,0.264511,12956,0.044472,1.202108
NE,0.184781,10802,-0.035258,0.839763
NNE,0.223929,9896,0.003890,1.017678
NNW,0.276378,9686,0.056339,1.256042
NW,0.279713,11973,0.059674,1.271196



WindDir9am


,mean,count,diff,risk
WindDir9am,,,,
E,0.140697,13888,-0.079342,0.639418
ENE,0.167546,11758,-0.052493,0.761435
ESE,0.141881,11707,-0.078158,0.644799
Missing,0.190520,16140,-0.029519,0.865849
N,0.306819,16557,0.086780,1.394384
NE,0.205693,11172,-0.014346,0.934802
NNE,0.266938,12044,0.046899,1.213139
NNW,0.304634,11394,0.084595,1.384455
NW,0.286566,12074,0.066527,1.302343



WindDir3pm


,mean,count,diff,risk
WindDir3pm,,,,
E,0.165755,12066,-0.054284,0.753298
ENE,0.176486,11525,-0.043553,0.802066
ESE,0.165758,12470,-0.054281,0.753311
Missing,0.265142,9229,0.045103,1.204980
N,0.271212,12658,0.051173,1.232563
NE,0.191941,12259,-0.028098,0.872303
NNE,0.245124,9946,0.025085,1.114001
NNW,0.278031,11398,0.057992,1.263554
NW,0.282162,12227,0.062123,1.282329



RainToday


,mean,count,diff,risk
RainToday,,,,
Missing,0.431420,2056,0.211381,1.960654
No,0.148488,165266,-0.071551,0.674825
Yes,0.464830,46531,0.244791,2.112489


Some observations:

- If it rained today (`RainToday = Yes`), the chance of rain tomorrow is about 2 times higher than average.
- Location matters a lot: dry inland places (`Woomera`, `AliceSprings`, `Uluru`) have a risk around 0.3, while coastal places such as `Portland` and `Walpole` have a risk around 1.6.
- Northerly and westerly winds (`N`, `NNW`, `NW`, `W`) are linked to more rain, easterly winds (`E`, `ESE`) to less rain.

### 4.4 Feature importance: mutual information (categorical features)

In [18]:
def mutual_info_rain_score(series):
    return mutual_info_score(series, df_full_train[target])


mi = df_full_train[categorical].apply(mutual_info_rain_score)
mi.sort_values(ascending=False)

RainToday      0.045469
Location       0.014777
WindDir9am     0.008806
WindGustDir    0.005256
WindDir3pm     0.004522
dtype: float64

`RainToday` is the most informative categorical feature, followed by `Location`.

### 4.5 Feature importance: correlation (numerical features)

In [19]:
df_full_train[numerical].corrwith(df_full_train[target]).abs().sort_values(ascending=False)

RISK_MM          0.496396
Humidity3pm      0.439346
Humidity9am      0.254592
Rainfall         0.238985
WindGustSpeed    0.227590
Pressure9am      0.224517
Pressure3pm      0.201813
Temp3pm          0.191660
MaxTemp          0.160624
WindSpeed9am     0.091338
WindSpeed3pm     0.085742
MinTemp          0.085051
Temp9am          0.026485
dtype: float64

`RISK_MM` has the highest correlation, but it is the amount of rain that falls **tomorrow**, the quantity we want to predict. `RainTomorrow` is defined from it (`RISK_MM > 1 mm`), so it would not be available at prediction time. Using it would be **data leakage** and give unrealistically good results, so we remove it.

Among the real features, `Humidity3pm` is the strongest one, followed by `Humidity9am`, `Rainfall`, `WindGustSpeed` and the pressures.

In [20]:
numerical = [c for c in numerical if c != 'RISK_MM']
features = categorical + numerical
features

['Location',
 'WindGustDir',
 'WindDir9am',
 'WindDir3pm',
 'RainToday',
 'MinTemp',
 'MaxTemp',
 'Rainfall',
 'WindGustSpeed',
 'WindSpeed9am',
 'WindSpeed3pm',
 'Humidity9am',
 'Humidity3pm',
 'Pressure9am',
 'Pressure3pm',
 'Temp9am',
 'Temp3pm']

## 5. Feature encoding

We use `DictVectorizer` to one-hot encode the categorical features. Numerical features are passed through unchanged.

In [21]:
dv = DictVectorizer(sparse=False)

train_dicts = df_train[features].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

valid_dicts = df_valid[features].to_dict(orient='records')
X_valid = dv.transform(valid_dicts)

X_train.shape, X_valid.shape

((160389, 115), (53464, 115))

## 6. Training models

We train each model on `X_train` and evaluate it on `X_valid` using accuracy and ROC AUC. The helper below avoids repeating the same evaluation code, and `results` collects the scores for the final comparison.

In [22]:
results = {}


def evaluate(model, X, y):
    y_prob = model.predict_proba(X)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        'accuracy': accuracy_score(y_true=y, y_pred=y_pred),
        'roc_auc': roc_auc_score(y_true=y, y_score=y_prob),
    }

### 6.1 Baseline

A "model" that always predicts *no rain*. Any useful model has to beat it.

In [23]:
y_baseline = np.zeros_like(y_valid)

results['Baseline (always No)'] = {
    'accuracy': accuracy_score(y_true=y_valid, y_pred=y_baseline),
    'roc_auc': roc_auc_score(y_true=y_valid, y_score=y_baseline),
}
results['Baseline (always No)']

{'accuracy': 0.7794590752655993, 'roc_auc': 0.5}

### 6.2 Logistic Regression

The default `lbfgs` solver did not converge in 100 iterations on this unscaled data, so we use the `liblinear` solver and allow more iterations.

In [24]:
lr_model = LogisticRegression(solver='liblinear', max_iter=1000, random_state=1)
lr_model.fit(X_train, y_train)

results['Logistic Regression'] = evaluate(lr_model, X_valid, y_valid)
results['Logistic Regression']

{'accuracy': 0.8499364058057759, 'roc_auc': 0.8635064693472504}

### 6.3 Decision Tree

A decision tree without a depth limit fits the training data perfectly and overfits. We tune `max_depth` on the validation set.

In [25]:
dt_full = DecisionTreeClassifier(random_state=1)
dt_full.fit(X_train, y_train)

print('No depth limit:')
print('  train:', evaluate(dt_full, X_train, y_train))
print('  valid:', evaluate(dt_full, X_valid, y_valid))

No depth limit:
  train: {'accuracy': 0.9999812954753755, 'roc_auc': 0.999999998866855}
  valid: {'accuracy': 0.787614095466108, 'roc_auc': 0.6959686776918703}


In [26]:
dt_scores = {}

for depth in [1, 2, 3, 5, 7, 10, 12, 15, 20]:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=1)
    dt.fit(X_train, y_train)
    dt_scores[depth] = evaluate(dt, X_valid, y_valid)['roc_auc']
    print(f'max_depth={depth:>2}  auc={dt_scores[depth]:.4f}')

best_depth = max(dt_scores, key=dt_scores.get)
print('best max_depth:', best_depth)

max_depth= 1  auc=0.6912
max_depth= 2  auc=0.7734
max_depth= 3  auc=0.8033
max_depth= 5  auc=0.8330
max_depth= 7  auc=0.8469
max_depth=10  auc=0.8466
max_depth=12  auc=0.8264
max_depth=15  auc=0.7739
max_depth=20  auc=0.6961
best max_depth: 7


In [27]:
dt_model = DecisionTreeClassifier(max_depth=best_depth, random_state=1)
dt_model.fit(X_train, y_train)

results['Decision Tree'] = evaluate(dt_model, X_valid, y_valid)
results['Decision Tree']

{'accuracy': 0.8462142750261858, 'roc_auc': 0.8469206925310306}

### 6.4 Random Forest

In [28]:
rf_model = RandomForestClassifier(n_estimators=200, random_state=1, n_jobs=-1)
rf_model.fit(X_train, y_train)

results['Random Forest'] = evaluate(rf_model, X_valid, y_valid)
results['Random Forest']

{'accuracy': 0.8605042645518479, 'roc_auc': 0.885062108334107}

### 6.5 XGBoost

XGBoost was added to the project with `uv add xgboost`. We use its scikit-learn interface (`XGBClassifier`).

In [29]:
xgb_model = XGBClassifier(n_estimators=200, learning_rate=0.3, random_state=1, n_jobs=-1)
xgb_model.fit(X_train, y_train)

results['XGBoost'] = evaluate(xgb_model, X_valid, y_valid)
results['XGBoost']

{'accuracy': 0.8657227293131827, 'roc_auc': 0.8936239401728825}

XGBoost also has a native API (`xgb.train` with `DMatrix`). With the same parameters it gives the same result. We keep the scikit-learn interface in the rest of the project because it is simpler and consistent with the other models.

In [30]:
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

feature_names = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_names)
dvalid = xgb.DMatrix(X_valid, label=y_valid, feature_names=feature_names)

xgb_native = xgb.train(xgb_params, dtrain, num_boost_round=200)

y_prob = xgb_native.predict(dvalid)
y_pred = (y_prob >= 0.5).astype(int)

print(f'Accuracy: {accuracy_score(y_valid, y_pred):.4f}')
print(f'ROC AUC:  {roc_auc_score(y_valid, y_prob):.4f}')

Accuracy: 0.8657
ROC AUC:  0.8936


## 7. Model comparison

In [31]:
df_results = pd.DataFrame(results).T.sort_values('roc_auc', ascending=False)
df_results.round(4)

,accuracy,roc_auc
XGBoost,0.8657,0.8936
Random Forest,0.8605,0.8851
Logistic Regression,0.8499,0.8635
Decision Tree,0.8462,0.8469
Baseline (always No),0.7795,0.5000


XGBoost gets the best validation scores (accuracy and ROC AUC), followed by Random Forest. All models clearly beat the baseline, so we choose **XGBoost** as the final model.

## 8. Final model

We retrain XGBoost with the same parameters on `df_full_train` (train + validation, 80% of the data) and evaluate it **once** on `df_test`, which no model has seen so far.

Both `df_full_train` and `df_test` were already filled with `fill_missing()` in section 3.1 (training medians and `'Missing'`), so they contain no missing values.

In [32]:
full_train_dicts = df_full_train[features].to_dict(orient='records')
test_dicts = df_test[features].to_dict(orient='records')

# a new DictVectorizer, so the `dv` fitted on df_train above stays unchanged
dv_full = DictVectorizer(sparse=False)
X_full_train = dv_full.fit_transform(full_train_dicts)
X_test = dv_full.transform(test_dicts)

X_full_train.shape, X_test.shape

((213853, 115), (53464, 115))

In [33]:
best_model = XGBClassifier(n_estimators=200, learning_rate=0.3, random_state=1, n_jobs=-1)
best_model.fit(X_full_train, y_full_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [34]:
test_scores = evaluate(best_model, X_test, y_test)

pd.DataFrame({
    'validation (trained on df_train)': results['XGBoost'],
    'test (trained on df_full_train)': test_scores,
}).round(4)

,validation (trained on df_train),test (trained on df_full_train)
accuracy,0.8657,0.8634
roc_auc,0.8936,0.8935


The test scores are very close to the validation scores, so the model generalizes well to unseen data and the validation results were not overly optimistic.

### 8.1 Saving the model

To make predictions in the web service we need everything that turns a raw weather record into a prediction:

- `train_medians` and the `categorical` columns, to fill missing values exactly like `fill_missing()`
- `dv_full`, to encode the record
- `best_model`, to predict

We save them together in one file with `pickle`.

In [35]:
model_file = 'weather_forecast.bin'

artifacts = {
    'train_medians': train_medians,
    'categorical': categorical,
    'dv': dv_full,
    'model': best_model,
}

with open(model_file, 'wb') as f_out:
    pickle.dump(artifacts, f_out)

print(f'Saved to {model_file}')

Saved to weather_forecast.bin


### 8.2 Checking the saved model

We load the file and predict a raw record from the test set (before filling missing values), the same way the web service will. The result must match the prediction of `best_model` in this notebook.

In [36]:
with open(model_file, 'rb') as f_in:
    loaded = pickle.load(f_in)

# a raw test record, still with its missing values (if any)
record = df_clean.loc[df_test.index[0], features].to_dict()
record

{'Location': 'Launceston',
 'WindGustDir': 'SE',
 'WindDir9am': 'ENE',
 'WindDir3pm': 'N',
 'RainToday': 'No',
 'MinTemp': 0.0,
 'MaxTemp': 13.0,
 'Rainfall': 0.0,
 'WindGustSpeed': 22.0,
 'WindSpeed9am': 7.0,
 'WindSpeed3pm': 6.0,
 'Humidity9am': 81.0,
 'Humidity3pm': 49.0,
 'Pressure9am': 1020.2,
 'Pressure3pm': 1017.3,
 'Temp9am': 6.2,
 'Temp3pm': 12.4}

In [37]:
def predict_rain(record, artifacts):
    df_record = pd.DataFrame([record])
    df_record = df_record.fillna(artifacts['train_medians'])
    df_record[artifacts['categorical']] = df_record[artifacts['categorical']].fillna('Missing')

    X = artifacts['dv'].transform(df_record.to_dict(orient='records'))
    return float(artifacts['model'].predict_proba(X)[0, 1])


prob_loaded = predict_rain(record, loaded)
prob_notebook = best_model.predict_proba(X_test[:1])[0, 1]

print(f'Rain probability (loaded model):   {prob_loaded:.4f}')
print(f'Rain probability (notebook model): {prob_notebook:.4f}')
print('Actual RainTomorrow:', y_test[0])

Rain probability (loaded model):   0.0373
Rain probability (notebook model): 0.0373
Actual RainTomorrow: 0


The loaded model gives the same prediction, so the saved file is ready to be used.

**Next steps:** serve the model as a web service (section 9), package it with Docker (section 10) and deploy it to AWS Elastic Beanstalk (section 11).

## 9. Serving the model as a web service

So far the model only lives inside this notebook. To make it useful for other programs, we put it behind a **web service**: a small program that waits for HTTP requests, runs the model and sends back the answer.

The plan is always the same, and every step is tested before moving on:

1. **Local Python:** `predict.py` runs on the laptop and `test.py` sends it a request.
2. **Docker:** the same service runs inside a container on the laptop.
3. **Cloud:** the same container runs on AWS Elastic Beanstalk.

> **About the commands in sections 9–11.** They are shell commands, written in markdown blocks so they **don't run** when the notebook is executed. You normally type them in a **terminal** (PowerShell) in the project folder.
>
> They *can* also be run from a notebook code cell by starting the line with `!`, for example `!docker build -t rain-prediction .` or `!eb status`. The `!` tells Jupyter "run this in the shell, not in Python". Be careful with three kinds of commands:
>
> - **Commands that never finish**, like `python predict.py` or `docker run -it ...`: they are servers that keep running, so the cell would stay busy forever and block the notebook. Run them in a terminal instead, or use `docker run -d` (see 10.3).
> - **Commands that ask a question**, like `eb terminate`, which waits for you to type the environment name: a notebook cell can't answer, so it hangs. Use the flags shown below that skip the question (for example `eb terminate --force`).
> - **Commands that cost money**, like `eb create`: running that cell by accident starts a paid AWS machine.

### 9.1 The web service: `predict.py`

We use [FastAPI](https://fastapi.tiangolo.com/) to build the web service and **uvicorn** to run it (uvicorn is the server program that actually listens for requests and passes them to FastAPI). `requests` is used by `test.py` to send requests. All three were added with:

```bash
uv add fastapi uvicorn requests
```

`predict.py`:

```python
import pickle
import pandas as pd

from fastapi import FastAPI

app = FastAPI()

with open('weather_forecast.bin', 'rb') as f_in:
    artifacts = pickle.load(f_in)


def predict_rain(record, artifacts):
    df_record = pd.DataFrame([record])
    df_record = df_record.reindex(columns=list(artifacts['train_medians'].index) + artifacts['categorical'])
    df_record = df_record.fillna(artifacts['train_medians'])
    df_record[artifacts['categorical']] = df_record[artifacts['categorical']].fillna('Missing')

    X = artifacts['dv'].transform(df_record.to_dict(orient='records'))
    probability = float(artifacts['model'].predict_proba(X)[0, 1])

    return probability

@app.post("/predict")
def predict(record: dict):
    probability = predict_rain(record, artifacts)
    return {
        "rain_probability": probability,
        "rain_tomorrow": "yes" if probability >= 0.5 else "no",
    }


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=9696)
```

How it works:

- **The model is loaded once**, when the service starts, not on every request.
- **`predict_rain()` is the same function as in section 8.2**, plus one line: `reindex(...)`. If a client leaves out a field (for example `WindGustDir`), its column doesn't exist and `df_record[artifacts['categorical']]` would crash with a `KeyError`. `reindex` adds every missing column as an empty value, so the `fillna` lines fill it like any other missing value. It also drops unknown extra fields.
- **The endpoint is a separate small function.** `@app.post("/predict")` means: when a `POST` request arrives at the address `/predict`, call `predict()`.
- **`record: dict` matters.** FastAPI decides where each parameter comes from by its type hint. With `record: dict`, it reads the **JSON body** of the request. Without a type hint, FastAPI treats a parameter as a **query parameter** in the URL (`/predict?record=...`), and a JSON request fails with `422 Unprocessable Entity`. That's also why the endpoint can't have an `artifacts` parameter: the client can't send the model.
- **It returns a dict**, which FastAPI converts to JSON: `{"rain_probability": 0.0373, "rain_tomorrow": "no"}`. A dict with names is clearer for clients than a tuple, which would arrive as an unlabeled list.
- **`if __name__ == "__main__":`** runs only when the file is started directly (`python predict.py`), not when another program imports it (like uvicorn in Docker). Test code at the top level of the file would run every time the server starts, so it belongs in `test.py`.
- **`host="0.0.0.0"`** tells uvicorn to accept connections from anywhere, not only from the same machine. This is needed in Docker (see 10.2).

### 9.2 Running and testing the service locally

Start the service in a terminal (it keeps running until you press `Ctrl+C`):

```bash
uv run python predict.py
```

In a **second** terminal, run `test.py`, which sends one raw weather record (the test record from section 8.2) to the service:

```python
import requests

record = {"Location": "Launceston", "WindGustDir": "SE", "WindDir9am": "ENE", "WindDir3pm": "N",
          "RainToday": "No", "MinTemp": 0.0, "MaxTemp": 13.0, "Rainfall": 0.0,
          "WindGustSpeed": 22.0, "WindSpeed9am": 7.0, "WindSpeed3pm": 6.0,
          "Humidity9am": 81.0, "Humidity3pm": 49.0, "Pressure9am": 1020.2, "Pressure3pm": 1017.3,
          "Temp9am": 6.2, "Temp3pm": 12.4}

url = "http://localhost:9696/predict"   # local (python predict.py or docker run)
# url = "http://<your-environment>.elasticbeanstalk.com/predict"   # AWS Elastic Beanstalk (section 11)

response = requests.post(url, json=record)

print(response)
print(response.json())
```

```bash
uv run python test.py
```

Expected output (the same probability as in section 8.2):

```
<Response [200]>
{'rain_probability': 0.03729541599750519, 'rain_tomorrow': 'no'}
```

**Testing in the browser.** FastAPI automatically builds an interactive test page at **http://localhost:9696/docs**. Click **POST /predict**, then **Try it out**, paste a record as JSON and click **Execute**.

Things that are easy to get wrong:

- **Open `localhost`, not `0.0.0.0`.** `0.0.0.0` in `host="0.0.0.0"` is a *listening* setting ("accept connections from anywhere"), not an address you can visit. To reach a service on your own machine, use `localhost` (or `127.0.0.1`).
- **`http://localhost:9696/` alone shows `{"detail":"Not Found"}`.** That's normal: the app only has the `/predict` route. Opening `/predict` in the browser gives "Method Not Allowed", because the browser sends a `GET` request and the route only accepts `POST`.
- **Only one program can use a port.** If an old server is still running on port 9696, a new one fails to start, or worse, you test the old code by mistake. On Windows, this shows which process uses the port (PowerShell):

```powershell
Get-NetTCPConnection -LocalPort 9696 -State Listen | ForEach-Object { Get-Process -Id $_.OwningProcess }
```

## 10. Packaging the service with Docker

The service runs on the laptop because Python 3.11, pandas, xgboost and the rest are installed in `.venv`. A server in the cloud has none of that. **Docker** packages the code together with everything it needs to run, so it behaves the same everywhere.

### 10.1 Key ideas

- **Dockerfile:** the recipe, a list of steps to build the package (run from top to bottom).
- **Image:** the finished package, built from the recipe with `docker build`.
- **Container:** a running copy of the image, started with `docker run`.

A container is like a small, separate Linux computer that only has what the recipe put in it. It **can't see your files** unless you `COPY` them in, and **nobody can reach it** from outside unless you publish a port.

Docker Desktop must be running ("Engine running") before any `docker` command works.

### 10.2 The Dockerfile

The project's `Dockerfile` has a comment above every line. Without the comments it is:

```dockerfile
FROM python:3.11-slim

RUN pip install uv

WORKDIR /app

COPY pyproject.toml uv.lock ./

RUN uv sync --locked --no-install-project

COPY predict.py weather_forecast.bin ./

ENV PATH="/app/.venv/bin:$PATH"

EXPOSE 9696

CMD ["uvicorn", "predict:app", "--host", "0.0.0.0", "--port", "9696"]
```

| Instruction | What it does | Here |
|---|---|---|
| `FROM` | The starting image everything is built on. | A small Debian Linux with Python 3.11. Must match `.python-version`; `slim` leaves out tools we don't need. |
| `RUN` | Runs a shell command **while the image is built**. | Installs `uv`; then installs the dependencies. |
| `WORKDIR` | Creates a folder and works from it in all later steps (like `cd`). | `/app`, so `./` means `/app`. |
| `COPY` | Copies files from the laptop into the image. | First the dependency files, later the code and model. |
| `ENV` | Sets an environment variable. | Puts `/app/.venv/bin` first on `PATH`, the same as `activate`. |
| `EXPOSE` | Records which port the app listens on. It does **not** open the port by itself. | `9696`. `docker run -p` opens it locally; Elastic Beanstalk reads this line. |
| `CMD` | The command that runs **when the container starts** (not during the build). | Starts uvicorn with the `app` object from `predict.py`. |

Details worth remembering:

- **`uv sync --locked --no-install-project`:** `--locked` installs the exact versions from `uv.lock` (the same as on the laptop) and fails instead of silently updating. `--no-install-project` installs only the libraries, not our own `src/` package, which the service doesn't need. (Because `pyproject.toml` has a `[build-system]` section, a plain `uv sync` would try to build the project and need `src/` and `README.md` in the image.)
- **Why copy in two steps (layer caching).** Docker saves the result of each step as a **layer** and reuses it if the step's inputs haven't changed. Installing xgboost and pandas is slow, so the dependency files are copied and installed **before** `predict.py`. Editing `predict.py` then only re-runs the last fast steps instead of reinstalling everything.
- **Only the needed files are copied.** The CSV, notebooks and `.venv` stay out, which keeps the image smaller. (`.venv` must never be copied: it contains Windows packages, and the container runs Linux.)
- **`--host 0.0.0.0` is required in Docker.** By default uvicorn only accepts connections from inside the container itself (`127.0.0.1`), so requests from the laptop would never arrive.

**Reusing this Dockerfile for another project:** usually you only change the Python version in `FROM`, the files in the second `COPY`, the port in `EXPOSE` and `CMD`, and the `module:app` name in `CMD`.

### 10.3 Building and running the container

Build the image from the project folder (`-t` gives the image a name, `.` means "use the Dockerfile in this folder"):

```bash
docker build -t rain-prediction .
```

Run a container from the image:

```bash
docker run -it --rm -p 9696:9696 rain-prediction
```

- `-p 9696:9696` connects port 9696 on the laptop to port 9696 in the container (`laptop_port:container_port`). Without it, `test.py` can't reach the container.
- `--rm` deletes the container when it stops.
- `-it` shows the output in the terminal and lets `Ctrl+C` stop it.

Then test it exactly like in 9.2: `uv run python test.py`, or open http://localhost:9696/docs. The answer must be the same 0.0373. (Stop any `python predict.py` server first; both would use port 9696.)

Useful commands:

```bash
docker ps                      # list running containers (name, ports, status)
docker stop <container-name>   # stop a container (name from docker ps)
docker images                  # list images and their sizes
```

From a **notebook cell**, run the container in the background with `-d` (detached) and give it a name, so the cell finishes and you can stop it later:

```
!docker run -d --rm -p 9696:9696 --name rain-prediction rain-prediction
!docker stop rain-prediction
```

The image is about 1.9 GB on disk, mostly from xgboost, scikit-learn and their dependencies. That is normal and fine for Elastic Beanstalk.

## 11. Deploying to AWS Elastic Beanstalk

**Elastic Beanstalk (EB)** is an AWS service that runs our Docker image in the cloud. We give it the project folder and it:

1. rents a virtual machine (an **EC2 instance**) in an AWS data center,
2. installs Docker on it,
3. builds the image from our `Dockerfile` and runs the container,
4. gives us a public URL that sends web traffic to the port in our `EXPOSE` line.

EB uses two words you'll see a lot:

- **Application:** the project as a whole, here `rain-prediction`. It's only a name and a folder for your settings, so it **costs nothing**.
- **Environment:** a running copy of the application, meaning the actual machine and URL. **This is the part that costs money**, and it's what we delete at the end (11.5).

**Prerequisites** (done once per computer):

- An AWS account and an IAM user with access keys.
- The **AWS CLI** (`aws` command), configured with `aws configure` (access key, secret key, default region). `aws sts get-caller-identity` shows which user you're logged in as.
- The **EB CLI** (`eb` command), installed for example with `uv tool install awsebcli` (or `pip install awsebcli`). `eb --version` checks it.

All `eb` commands are run from the **project folder**.

### 11.1 Creating the application: `eb init`

```bash
eb init -p docker -r eu-north-1 rain-prediction
```

- `-p docker` sets the **platform**: it tells EB this is a Docker app, so EB runs our `Dockerfile` instead of setting up plain Python.
- `-r eu-north-1` is the **region**: which AWS data center to use (here Stockholm, the same as the AWS CLI default).
- `rain-prediction` is the **application name**.

This only registers the application with AWS. No machine starts, so it's free. Output: `Application rain-prediction has been created.`

If it asks questions (in a terminal): answer **n** to **CodeCommit**; for **SSH**, either answer works, **n** is simpler. With `-p` and `-r` given, it normally doesn't ask anything, so it also works from a notebook cell with `!eb init ...`.

**What it creates:** a hidden folder `.elasticbeanstalk/` with `config.yml`. That file is where `eb` remembers the application name, platform and region, so later commands don't need those flags again:

```yaml
global:
  application_name: rain-prediction
  default_platform: Docker
  default_region: eu-north-1
  ...
```

It also creates a `.gitignore` file, which keeps EB's private settings out of Git. (It only matters if the project is a Git repository.)

### 11.2 Choosing what gets uploaded: `.ebignore`

When you deploy, the EB CLI **zips the project folder and uploads it to AWS**, and AWS runs `docker build` on the machine. Without instructions it zips **everything**: the 30 MB CSV, `.venv` (hundreds of MB of Windows-only packages, useless on AWS's Linux machine), notebooks, backups...

A **`.ebignore`** file lists what to leave out of the zip. It works like `.gitignore`: one pattern per line, and `!` at the start means "but *do* include this". The safest approach is an **allow list**: ignore everything, then allow only the files the Docker build needs.

```
# Ignore everything...
*

# ...except the files needed to build and run the Docker image
# (rule: every file the Dockerfile COPYs must be allowed here)
!/Dockerfile
!/pyproject.toml
!/uv.lock
!/predict.py
!/weather_forecast.bin
```

With this file the upload is only these 5 files, about 0.36 MB. New files added to the project later (notebooks, data) stay out automatically.

Lessons learned:

- **Rule:** if the `Dockerfile` copies a file, `.ebignore` must allow it, or the build on AWS fails with "file not found".
- **Location:** `.ebignore` must be in the **project root**, next to the `Dockerfile`, *not* inside `.elasticbeanstalk/`. In the wrong place it is silently ignored and the whole folder is uploaded.
- **Leading `/`:** `!predict.py` matches a file with that name in **any** folder (it pulled in `.venv/.../xgboost/testing/predict.py` and pandas' own `pyproject.toml`). `!/predict.py` means "only the one in the project root".

### 11.3 Creating the environment: `eb create`

> **Costs start here** (a few cents per hour) and continue until the environment is deleted in 11.5.

```bash
eb create rain-prediction-env --single
```

- `rain-prediction-env` is the **environment name**. It also becomes part of the public URL.
- **`--single`** gives **one machine with no load balancer**. By default, EB adds a load balancer: a separate AWS service that spreads traffic across several machines. It costs extra (roughly $15–20 a month even with almost no traffic) and isn't needed for a one-person project.

What happens:

1. "Creating application version archive" and "Uploading": the CLI zips the 5 files and uploads them to an **S3 bucket** (AWS's file storage). EB keeps every uploaded version there.
2. EB starts an EC2 machine, installs Docker, runs `docker build` with our `Dockerfile` and starts the container.
3. Event messages stream by. This usually takes **5–10 minutes**; the first build has to install xgboost and pandas.
4. At the end:

```
INFO    Instance deployment completed successfully.
INFO    Application available at rain-prediction-env.eba-piacx6xp.eu-north-1.elasticbeanstalk.com.
INFO    Successfully launched environment: rain-prediction-env
```

The message "An update to the EB CLI is available" can be ignored; the installed version works.

Useful commands while the environment exists:

```bash
eb status      # status (Ready), health (Green), URL (CNAME), platform
eb health      # more detailed health information
eb logs        # logs from the machine, e.g. Docker build errors or Python tracebacks
eb open        # opens the environment's URL in the browser
eb deploy      # uploads the current code again and redeploys it (after changing predict.py, for example)
```

### 11.4 Testing the deployed service

The service is now reachable from anywhere on the internet. Note that the URL has **no `:9696`**:

- browser: **http://rain-prediction-env.eba-piacx6xp.eu-north-1.elasticbeanstalk.com/docs**
- `test.py`: switch the `url` line to the EB address:

```python
# url = "http://localhost:9696/predict"   # local (python predict.py or docker run)
url = "http://rain-prediction-env.eba-piacx6xp.eu-north-1.elasticbeanstalk.com/predict"   # AWS Elastic Beanstalk
```

Result of the deployed service (the same as in the notebook, locally, and in local Docker, so the model in the cloud behaves exactly like the one on the laptop):

```
<Response [200]>
{'rain_probability': 0.03729541599750519, 'rain_tomorrow': 'no'}
```

**Why there is no port in the URL:** websites use port 80 by default. On the EB machine, a web server called **nginx** listens on port 80 and forwards each request to the container's port 9696, the port from the `EXPOSE` line. That's why `EXPOSE` matters on EB.

For the project, we took screenshots as proof that the service ran on AWS.

**The environment in the AWS console** (Elastic Beanstalk → Environments): health *Ok*, platform Docker, and the domain that became the public URL.

![Elastic Beanstalk environment in the AWS console](images/environment-aws-screenshot.png)

**The `/docs` page of the deployed service:** the request went to the `elasticbeanstalk.com` URL and the answer is the same 0.0373. The response header `server: nginx` shows the request passed through nginx on port 80 before reaching the container.

![Response of the deployed service on the /docs page](images/ev-server-response.png)

**`test.py` sending the request to the deployed service:**

![Output of test.py against the deployed service](images/test-script-response.png)

(The environment above has been deleted, so this URL no longer works. A new `eb create` gives a new URL.)

### 11.5 Deleting the environment: `eb terminate`

When you're done, delete the environment so it stops costing money:

```bash
eb terminate rain-prediction-env
```

- For safety, it asks you to **type the environment name again** to confirm.
- It takes about **3–5 minutes** and ends with `terminateEnvironment completed successfully`.
- It deletes everything that costs money: the EC2 machine, its public IP address, and the **security group** (the firewall rules EB created).
- It **can't be undone**; the URL stops working. To deploy again later, just run `eb create` again: the files and `.elasticbeanstalk/config.yml` are still on the laptop.

From a **notebook cell**, the confirmation question would hang the cell, so skip it with `--force` (this deletes immediately, so double-check the name):

```
!eb terminate rain-prediction-env --force
```

**What stays after terminating** (free or practically free):

- The **application** `rain-prediction` (only a name and settings).
- The **uploaded versions** (the 0.36 MB zips) in an S3 bucket named `elasticbeanstalk-<region>-<account-id>`. Storage this small costs a fraction of a cent per month.

**Checking that nothing is left running** (AWS CLI):

```bash
# the environment should show "Terminated" (Grey)
aws elasticbeanstalk describe-environments --application-name rain-prediction --query "Environments[].[EnvironmentName,Status,Health]" --output table

# EC2 machines that are not terminated: should be empty
aws ec2 describe-instances --filters "Name=instance-state-name,Values=pending,running,stopping,stopped" --query "Reservations[].Instances[].[InstanceId,State.Name,InstanceType]" --output table

# Elastic IPs (unused ones are billed) and load balancers: should be empty
aws ec2 describe-addresses --query "Addresses[].[PublicIp,InstanceId]" --output table
aws elbv2 describe-load-balancers --query "LoadBalancers[].LoadBalancerName" --output table
```

**Optional: removing everything completely.** Delete the application, then its uploaded zips. The S3 bucket is shared by all EB applications in the region, so delete only this application's folder, not the whole bucket:

```bash
aws elasticbeanstalk delete-application --application-name rain-prediction --terminate-env-by-force

# find the bucket name, then delete only the rain-prediction/ folder in it
aws s3 ls
aws s3 rm s3://elasticbeanstalk-eu-north-1-<account-id>/rain-prediction/ --recursive
```

After that, you can also delete the local `.elasticbeanstalk/` folder.

### 11.6 Command summary

| Step | Command | Costs money? |
|---|---|---|
| Run the service locally | `uv run python predict.py` | no |
| Test it | `uv run python test.py`, or http://localhost:9696/docs | no |
| Build the Docker image | `docker build -t rain-prediction .` | no |
| Run the container | `docker run -it --rm -p 9696:9696 rain-prediction` | no |
| Create the EB application | `eb init -p docker -r eu-north-1 rain-prediction` | no |
| Choose the uploaded files | `.ebignore` in the project root | no |
| Create the environment | `eb create rain-prediction-env --single` | **yes, from now on** |
| Check / debug / redeploy | `eb status`, `eb logs`, `eb deploy` | (running) |
| Delete the environment | `eb terminate rain-prediction-env` | stops the costs |
| Check the cleanup | `aws elasticbeanstalk describe-environments ...`, `aws ec2 describe-instances ...` | no |

## 12. Other ways to deploy a model on AWS

Elastic Beanstalk is the simplest way to run an always-on web service, and our `predict.py` and `Dockerfile` work with it as they are. AWS has several other options:

| Method | How it works | When to use it |
|---|---|---|
| **Elastic Beanstalk** (used here) | Upload the project; AWS creates a machine, builds the Dockerfile and runs the container all the time. | Simplest "always-on" web service. Good for this project. |
| **AWS Lambda** (serverless) | Nothing runs until a request arrives; AWS starts the code for that request and then stops it. The Docker image goes to **ECR** (AWS's image storage), and **API Gateway** gives it a URL. | Rare or irregular traffic. You pay per request, so it's nearly free when idle. The first request after a quiet period is slow (a "cold start"). The code needs a Lambda **handler function** instead of FastAPI/uvicorn. |
| **Kubernetes on EKS** | A cluster of machines; Kubernetes manages many containers on them: scaling, restarting, rolling out updates. | Large production systems with many services. Too much for one model, and the cluster itself costs money every hour. |
| **EC2 directly** | Rent a plain machine, connect with SSH, install Docker and run `docker run` yourself. | Learning how things work underneath. You manage everything by hand. |
| **App Runner / ECS Fargate** | Give AWS the image (in ECR) and it runs it, without managing machines. | A middle ground between Elastic Beanstalk and Kubernetes. |

Switching to Lambda would mean rewriting `predict.py` around a Lambda handler and setting up ECR and API Gateway. The model file and `predict_rain()` could be reused unchanged.